# Williams Midfield Validation

Standard evaluation on front-runners in clean air is misleading. The real test is the chaotic midfield.

This notebook evaluates our tire-cliff prediction models specifically on **midfield constructors**
(Williams, Haas, Alpine, RB, Kick Sauber) under the hardest conditions:

1. **Dirty-air exposure** — stints with >5 cumulative laps within 1.5s of the car ahead
2. **Non-optimal tire strategies** — drivers who deviated from the race consensus compound sequence
3. **Variable weather** — races with rainfall transitions or large track temperature swings

We then deep-dive into 3–5 real midfield battles comparing model recommendations to actual team decisions.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Ensure the repo root is importable
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.training.base import (
    load_training_data,
    prepare_features,
    split_by_time,
    split_by_time_adaptive,
)
from src.training.metrics import evaluate_all
from src.training.midfield_validation import (
    MIDFIELD_CONSTRUCTORS_2024,
    evaluate_midfield_slices,
    filter_dirty_air_stints,
    filter_midfield,
    filter_non_optimal_strategy,
    filter_variable_weather,
    print_summary,
    run_case_studies,
)

plt.rcParams.update({"figure.figsize": (10, 6), "figure.dpi": 120})
pd.set_option("display.max_columns", 40)

## 1. Setup — Load Data and Model

Load the labeled dataset, apply the temporal train/test split, and load
the champion model (or train a fresh XGBoost baseline if no registered model exists).

In [ ]:
df = load_training_data()
train_df, val_df, test_df = split_by_time(df)

if train_df.empty and not df.empty:
    print("Canonical split empty — falling back to adaptive split")
    train_df, val_df, test_df = split_by_time_adaptive(df)

print(f"Train: {len(train_df):,} rows | Val: {len(val_df):,} rows | Test: {len(test_df):,} rows")
print(f"Test sessions: {sorted(test_df['session_id'].unique())}")

In [ ]:
X_train, y_train = prepare_features(train_df, fit_encoder=True)
X_val, y_val = prepare_features(val_df)
X_test, y_test = prepare_features(test_df)

# Try to load champion model; fall back to training a fresh XGBoost
try:
    from src.training.registry import get_production_model
    model = get_production_model()
    if model is None:
        raise RuntimeError("No production model registered")
    print(f"Loaded production model from registry")
except Exception as e:
    print(f"Could not load production model ({e}); training fresh XGBoost baseline")
    from src.training.baseline import DEFAULT_XGB_PARAMS, xgb_model_fn
    model = xgb_model_fn(
        X_train, y_train, DEFAULT_XGB_PARAMS,
        eval_set=(X_val, y_val) if len(X_val) else None,
        show_progress=True,
    )

## 2. Midfield Filtering

Filter the test set to midfield constructors and inspect the distribution of
teams, compounds, and dirty-air exposure.

In [ ]:
midfield_df = filter_midfield(test_df)
print(f"Midfield rows: {len(midfield_df):,} / {len(test_df):,} total test rows")

if "team" in midfield_df.columns and len(midfield_df) > 0:
    team_counts = midfield_df.groupby("team").agg(
        rows=("laps_to_cliff", "size"),
        stints=("stint_number", "nunique"),
    )
    display(team_counts)
else:
    print("No midfield data available. Check that 'team' column exists in labeled dataset.")

In [ ]:
if len(midfield_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Compound distribution
    if "compound" in midfield_df.columns:
        midfield_df["compound"].value_counts().plot.bar(ax=axes[0], color="steelblue")
        axes[0].set_title("Compound Distribution (Midfield)")
        axes[0].set_ylabel("Count")

    # Dirty-air histogram
    if "dirty_air_cumulative_laps" in midfield_df.columns:
        midfield_df["dirty_air_cumulative_laps"].plot.hist(
            bins=20, ax=axes[1], color="coral", edgecolor="black"
        )
        axes[1].axvline(x=5, color="red", linestyle="--", label="Threshold (5 laps)")
        axes[1].set_title("Dirty-Air Cumulative Laps (Midfield)")
        axes[1].set_xlabel("Cumulative laps in dirty air")
        axes[1].legend()

    plt.tight_layout()
    plt.show()

## 3. Slice Evaluation

Evaluate the model across all slices: full test set, midfield-only, dirty air,
non-optimal strategy, and variable weather.

In [ ]:
slice_results = evaluate_midfield_slices(model, test_df, X_test, y_test)
print_summary(slice_results, [])

In [ ]:
# Build a comparison DataFrame
rows = []
for name, sr in slice_results.items():
    rows.append({
        "Slice": name,
        "Rows": sr.n_rows,
        "Stints": sr.n_stints,
        "MAE": sr.metrics.get("mae", float("nan")),
        "Precision@3": sr.metrics.get("precision_at_3", float("nan")),
        "Strategy Accuracy": sr.metrics.get("strategy_accuracy", float("nan")),
    })
comparison_df = pd.DataFrame(rows).set_index("Slice")
display(comparison_df)

In [ ]:
# Bar chart comparison
metric_cols = ["MAE", "Precision@3", "Strategy Accuracy"]
plot_df = comparison_df[metric_cols].dropna(how="all")

if len(plot_df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    colors = ["#2196F3", "#4CAF50", "#FF9800", "#F44336", "#9C27B0"]

    for i, col in enumerate(metric_cols):
        vals = plot_df[col].dropna()
        if len(vals) > 0:
            vals.plot.barh(ax=axes[i], color=colors[:len(vals)])
            axes[i].set_title(col)
            axes[i].set_xlabel(col)

    plt.tight_layout()
    plt.show()

## 4. Case Studies

Select 3–5 real midfield battles and examine whether the model would have
recommended a better pit window than the team actually chose.

In [ ]:
case_studies = run_case_studies(model, test_df, X_test, y_test, n=5)
print(f"Generated {len(case_studies)} case studies\n")

for i, cs in enumerate(case_studies, 1):
    print(f"{'=' * 60}")
    print(f"Case Study {i}: {cs.session_id}")
    print(f"Teams: {', '.join(cs.teams)}")
    print(f"Drivers: {', '.join(cs.drivers)}")
    print(f"Strategies: {cs.compounds_used}")
    print(f"\n{'Driver':<10} {'Actual Pits':<20} {'Model Rec.':<15} {'MAE':<8}")
    print("-" * 55)
    for drv in cs.drivers:
        pits = cs.actual_pit_laps.get(drv, [])
        rec = cs.model_recommended_pit.get(drv)
        mae = cs.mae_per_driver.get(drv, float("nan"))
        rec_str = f"{rec:.1f}" if rec is not None else "N/A"
        print(f"{drv:<10} {str(pits):<20} {rec_str:<15} {mae:.2f}")
    print(f"\n{cs.summary}\n")

In [ ]:
# Lap-by-lap prediction vs actual for each case study
y_pred_all = model.predict(X_test)
test_idx_map = {idx: i for i, idx in enumerate(test_df.index)}

for i, cs in enumerate(case_studies, 1):
    sess = test_df[test_df["session_id"] == cs.session_id]
    if len(sess) == 0:
        continue

    midfield_sess = sess[sess["team"].isin(cs.teams)] if "team" in sess.columns else sess
    drivers = sorted(midfield_sess["driver_number"].unique())

    if len(drivers) == 0:
        continue

    n_drivers = len(drivers)
    fig, axes = plt.subplots(1, n_drivers, figsize=(5 * n_drivers, 4), squeeze=False)
    fig.suptitle(f"Case Study {i}: {cs.session_id}", fontsize=13)

    for j, drv in enumerate(drivers):
        drv_df = midfield_sess[midfield_sess["driver_number"] == drv].sort_values("stint_lap_number")
        positions = [test_idx_map[idx] for idx in drv_df.index if idx in test_idx_map]
        if not positions:
            continue

        laps = drv_df.loc[[test_df.index[p] for p in positions], "stint_lap_number"].values
        y_true_drv = y_test[positions]
        y_pred_drv = y_pred_all[positions]

        ax = axes[0][j]
        ax.plot(laps, y_true_drv, "o-", label="Actual", markersize=3)
        ax.plot(laps, y_pred_drv, "x--", label="Predicted", markersize=3)
        team = drv_df["team"].iloc[0] if "team" in drv_df.columns else "?"
        ax.set_title(f"Driver {drv} ({team})")
        ax.set_xlabel("Stint Lap")
        ax.set_ylabel("Laps to Cliff")
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

## 5. Failure Mode Analysis

Identify where the model fails worst on midfield stints and categorise failure modes.

In [ ]:
if len(midfield_df) > 0:
    midfield_positions = [test_idx_map[idx] for idx in midfield_df.index if idx in test_idx_map]
    mid_y_true = y_test[midfield_positions]
    mid_y_pred = y_pred_all[midfield_positions]
    mid_errors = np.abs(mid_y_true - mid_y_pred)

    error_df = midfield_df.loc[[test_df.index[p] for p in midfield_positions]].copy()
    error_df["abs_error"] = mid_errors
    error_df["signed_error"] = mid_y_pred - mid_y_true

    print("=" * 50)
    print("TOP 10 WORST PREDICTIONS (Midfield)")
    print("=" * 50)
    worst = error_df.nlargest(10, "abs_error")[
        ["session_id", "driver_number", "team", "stint_number",
         "stint_lap_number", "abs_error", "signed_error"]
    ]
    display(worst)

    # Categorise failure modes
    high_dirty = error_df["dirty_air_cumulative_laps"] > 5 if "dirty_air_cumulative_laps" in error_df.columns else pd.Series(False, index=error_df.index)
    is_censored = error_df["is_censored"] == 1

    print(f"\nMean abs error (all midfield):    {mid_errors.mean():.2f}")
    print(f"Mean abs error (dirty air):        {mid_errors[high_dirty.values].mean():.2f}" if high_dirty.any() else "No dirty-air stints")
    print(f"Mean abs error (censored stints):  {mid_errors[is_censored.values].mean():.2f}" if is_censored.any() else "No censored stints")
    print(f"Mean abs error (uncensored):       {mid_errors[~is_censored.values].mean():.2f}" if (~is_censored).any() else "No uncensored stints")

    # Error distribution
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(error_df["signed_error"], bins=30, color="steelblue", edgecolor="black", alpha=0.7)
    ax.axvline(x=0, color="red", linestyle="--")
    ax.set_title("Signed Prediction Error Distribution (Midfield)")
    ax.set_xlabel("Predicted - Actual (laps)")
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No midfield data available for failure analysis.")

## 6. Summary

Key findings for the model card:

In [ ]:
print("KEY FINDINGS")
print("=" * 50)
for name, sr in slice_results.items():
    if sr.n_rows == 0:
        print(f"- {name}: No data available")
        continue
    mae = sr.metrics.get('mae', float('nan'))
    p3 = sr.metrics.get('precision_at_3', float('nan'))
    print(f"- {name}: MAE={mae:.2f}, Precision@3={p3:.1%} ({sr.n_rows} rows, {sr.n_stints} stints)")

if len(case_studies) > 0:
    print(f"\nCase studies analysed: {len(case_studies)}")
    for cs in case_studies:
        avg_mae = np.nanmean(list(cs.mae_per_driver.values()))
        print(f"  - {cs.session_id} ({', '.join(cs.teams)}): avg MAE={avg_mae:.2f}")